# Week3_ex2_analysis - H-field Through Two Closed Rings, and the Rings' Own Eddy-Current Reaction

![Week3_ex2 result](Images/Week3_ex2_result_image.png)

A 400 mm copper wire carries 100 A at 10 Hz, and its ends touch the region's x faces, so it behaves like an infinite wire. Two closed copper rings (36-sided, 18-20 mm, 2 mm tall, no excitation) sit in the same plane at d = 300 mm (ring_1) and d = 100 mm (ring_2). H·n is integrated over a 20 mm disk on each ring.

This is a light check. Nothing unexpected shows up, so there are no hypothesis steps. The Phase = 0deg part is checked against the infinite-wire closed form. The Phase = 90deg part is where the rings show up: they're closed conductors, so current is induced in them and pushes back on the flux.

$$\int_{disk} \mathbf{H}\cdot\mathbf{n}\,dA = I\left(d-\sqrt{d^2-a^2}\right),\qquad \frac{X_{90^\circ}}{X_{0^\circ}} \approx \frac{\omega L_{ring}}{R_{ring}}$$

## Phase = 0deg: the wire's own field

The two H integrals were read once from a Data Table in the AEDT GUI. The disk areas come from the Fields Calculator (`Integrate(Surface(field_circle_N), 1)`), and both disks read 0.0012423314 m². That matches a regular 24-gon inscribed in the 20 mm circle to every printed digit. So the "true circle" disk is still a polygon once it's meshed, and `Integrate` multiplies by that meshed area. The FEM values are negative only because of the disk normal (+z), so the table compares magnitudes.

In [1]:
import numpy as np
import pandas as pd

# 1. Model constants and the closed form for an infinite wire
I_wire = 100.0                               # A
a = 20e-3                                    # disk radius, m
d = {"ring_1": 300e-3, "ring_2": 100e-3}     # ring center to wire, m
rings = ["ring_1", "ring_2"]

def H_integral_closed(dist):
    return I_wire * (dist - np.sqrt(dist**2 - a**2))    # A*m

# 2. Disk areas, read once from the Fields Calculator in the AEDT GUI: Integrate(Surface(field_circle_N), 1)
area_fem = {"ring_1": 0.0012423314164921, "ring_2": 0.0012423314164921}
area_24gon = 12 * a**2 * np.sin(np.pi / 12)

# 3. coil1_H / coil2_H, read once from a Data Table in the AEDT GUI (Setup1 : LastAdaptive)
H_0deg = {"ring_1": -0.065710, "ring_2": -0.199312}
H_90deg = {"ring_1": -0.000736, "ring_2": -0.002205}

rows = []
for k in rings:
    closed = H_integral_closed(d[k])
    area_ratio = area_fem[k] / (np.pi * a**2)
    rows.append({
        "ring": k,
        "d [mm]": d[k] * 1e3,
        "meshed area / regular 24-gon": area_fem[k] / area_24gon,
        "meshed area vs true circle [%]": (area_ratio - 1) * 100,
        "closed form [A*m]": closed,
        "FEM, Phase 0deg [A*m]": abs(H_0deg[k]),
        "FEM vs closed form [%]": (abs(H_0deg[k]) / closed - 1) * 100,
        "FEM vs closed form x meshed area [%]": (abs(H_0deg[k]) / (closed * area_ratio) - 1) * 100,
    })

pd.DataFrame(rows)

,ring,d [mm],meshed area / regular 24-gon,meshed area vs true circle [%],closed form [A*m],"FEM, Phase 0deg [A*m]",FEM vs closed form [%],FEM vs closed form x meshed area [%]
0,ring_1,300.0,1.0,-1.138407,0.066741,0.065710,-1.544639,-0.410909
1,ring_2,100.0,1.0,-1.138407,0.202041,0.199312,-1.350730,-0.214768


## Phase = 90deg: current induced in the rings

The wire carries a fixed 100 A at phase 0, so its own field has no 90deg part. Whatever shows up at Phase = 90deg comes from current induced in the closed rings. When R is much larger than ωL, that current is almost entirely in quadrature, and its own flux through the disk is a fraction ωL/R of the external flux. That fraction should be the same for both rings. They're identical, and only the external flux changes with distance.

R below comes from the 36-sided ring geometry and copper's conductivity. L comes from the thin-ring formula with the square cross-section replaced by its geometric mean distance (0.447 x side). Skin depth at 10 Hz is about 21 mm, far larger than the 2 mm cross-section, so the DC resistance is fine.

In [2]:
# 4. Ring R and L from geometry (36-sided, 18-20 mm circumradius, 2 mm tall, copper)
mu0 = 4 * np.pi * 1e-7
sigma_cu = 5.8e7                 # S/m, AEDT copper
omega = 2 * np.pi * 10
n_sides = 36
r_in, r_out, h = 18e-3, 20e-3, 2e-3
r_mid = (r_in + r_out) / 2

path = n_sides * 2 * r_mid * np.sin(np.pi / n_sides)      # centerline length
width = (r_out - r_in) * np.cos(np.pi / n_sides)          # thickness across the flat sides
R_ring = path / (sigma_cu * width * h)
L_ring = mu0 * r_mid * (np.log(8 * r_mid / (0.447 * (r_out - r_in))) - 2)
skin_depth = np.sqrt(2 / (omega * mu0 * sigma_cu))

pd.DataFrame({
    "R_ring [mOhm]": [R_ring * 1e3],
    "L_ring [nH]": [L_ring * 1e9],
    "wL/R [%]": [omega * L_ring / R_ring * 100],
    "skin depth [mm]": [skin_depth * 1e3],
})

,R_ring [mOhm],L_ring [nH],wL/R [%],skin depth [mm]
0,0.515881,74.873792,0.911926,20.898068


In [3]:
# 5. Phase 90deg / Phase 0deg for each ring vs the R-L estimate
estimate = omega * L_ring / R_ring

pd.DataFrame({
    "ring": rings,
    "Phase 90deg / Phase 0deg [%]": [H_90deg[k] / H_0deg[k] * 100 for k in rings],
    "estimate wL/R [%]": [estimate * 100] * 2,
    "measured / estimate": [H_90deg[k] / H_0deg[k] / estimate for k in rings],
})

,ring,Phase 90deg / Phase 0deg [%],estimate wL/R [%],measured / estimate
0,ring_1,1.120073,0.911926,1.228249
1,ring_2,1.106306,0.911926,1.213152


## Conclusion

At Phase = 0deg both rings land 1.35-1.55% below the closed form. Almost all of that is the meshed disk: the 20 mm "true circle" is a regular 24-gon with 1.14% less area, and `Integrate` carries that area straight into the result. After scaling the closed form by the meshed area, what's left is -0.41% for ring_1 and -0.21% for ring_2. That's a small, uniform offset of the size you'd expect from a 1% energy-error adaptive mesh, and the two rings don't differ enough to read a distance trend into it.

At Phase = 90deg the rings' own induced current shows up. The 90deg/0deg ratio is 1.12% for ring_1 and 1.11% for ring_2, even though ring_2 sees three times the flux. So this ratio is a property of the ring (ωL/R), not of the wire's field. The R-L estimate gives 0.91%, which is the right order, but the measured ratio is about 1.2x higher in both rings. There are two candidates, and neither is tested here. First, the disk runs out to the ring's outer edge (20 mm) while the estimate uses the centerline (19 mm), so the disk collects some of the ring's own flux that doesn't link the whole ring current. Second, the ring's own field is concentrated right at the conductor, which is where the disk edge sits, so any meshing error along the rim gets amplified in this small quadrature part.

Nothing here needs a closer look. The Phase 0 part matches the infinite-wire closed form once the meshed area is accounted for, and the Phase 90 part is the eddy-current reaction you'd expect from a closed copper ring.